# LeetCode 1790 — Check if One String Swap Can Make Strings Equal bằng Python

Link đề: https://leetcode.com/problems/check-if-one-string-swap-can-make-strings-equal/

Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

```python
class Solution:
    def areAlmostEqual(self, s1: str, s2: str) -> bool:

```

**Mục tiêu học:**

1. Hiểu phép **swap** (hoán đổi) hai ký tự trong chuỗi và vì sao chuỗi Python phải đổi sang `list` mới swap được.
2. Viết được lời giải **brute force** thử mọi cặp swap, thấy rõ chi phí `O(n³)`.
3. Rút ra insight: chỉ những vị trí **khác nhau** mới quan trọng — phải có đúng 0 hoặc 2 vị trí khác nhau.
4. Viết lời giải `O(n)` thu thập chỉ số khác nhau, trace từng bước, chứng minh tính đúng.
5. Biết thêm cách dùng `zip`, `Counter` và nhận dạng pattern "so sánh hai chuỗi theo vị trí".

## 1. Bài toán yêu cầu gì?

Cho hai chuỗi `s1`, `s2` **cùng độ dài**. Một **string swap** là: chọn hai chỉ số (có thể trùng nhau) trong **một** chuỗi và đổi chỗ hai ký tự tại đó.

Trả `True` nếu có thể làm hai chuỗi bằng nhau bằng **nhiều nhất một** lần swap trên **đúng một** chuỗi; ngược lại `False`.

| `s1` | `s2` | Kết quả | Vì sao |
|---|---|:---:|---|
| `"bank"` | `"kanb"` | `True` | swap `k` và `b` của `s2` → `"bank"` |
| `"attack"` | `"defend"` | `False` | khác quá nhiều chỗ |
| `"kelb"` | `"kelb"` | `True` | đã bằng nhau, không cần swap |
| `"abcd"` | `"dcba"` | `False` | khác 4 vị trí, 1 swap chỉ sửa tối đa 2 |
| `"aa"` | `"ac"` | `False` | khác 1 vị trí — swap không sửa được |

**Ràng buộc:** `1 <= len(s1), len(s2) <= 100`, `len(s1) == len(s2)`, chỉ gồm chữ thường.

Ghi chú: "hai chỉ số có thể trùng nhau" nghĩa là swap `i` với chính `i` — không đổi gì. Đó là lý do hai chuỗi đã bằng nhau vẫn trả `True`.

In [1]:
examples = [
    ("bank", "kanb", True),
    ("attack", "defend", False),
    ("kelb", "kelb", True),
]
for s1, s2, expected in examples:
    print(f"s1={s1!r:9} s2={s2!r:9} -> kỳ vọng {expected}")

s1='bank'    s2='kanb'    -> kỳ vọng True
s1='attack'  s2='defend'  -> kỳ vọng False
s1='kelb'    s2='kelb'    -> kỳ vọng True


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi là **bất biến (immutable)**

Không thể gán `s[0] = 'x'` cho chuỗi. Muốn "sửa" chuỗi phải tạo chuỗi mới, hoặc đổi sang `list` (danh sách — **có thể sửa**), sửa xong rồi ghép lại bằng `"".join(...)`.

In [2]:
s = "kanb"
try:
    s[0] = "b"
except TypeError as e:
    print("Lỗi:", e)

chars = list(s)          # ['k', 'a', 'n', 'b']
print("list:", chars)
chars[0] = "b"
print("sau khi sửa:", chars, "->", "".join(chars))

Lỗi: 'str' object does not support item assignment
list: ['k', 'a', 'n', 'b']
sau khi sửa: ['b', 'a', 'n', 'b'] -> banb


### 2.2. Swap hai phần tử bằng gán song song

Python cho phép `a, b = b, a`. Với list: `chars[i], chars[j] = chars[j], chars[i]`. Vế phải được tính **trước**, rồi mới gán, nên không cần biến tạm.

In [3]:
chars = list("kanb")
i, j = 0, 3
chars[i], chars[j] = chars[j], chars[i]
print("".join(chars))   # bank

x, y = 1, 2
x, y = y, x
print(x, y)             # 2 1

bank
2 1


### 2.3. `zip` — đi song song hai chuỗi

`zip(s1, s2)` trả về từng cặp `(s1[i], s2[i])`. Kết hợp `enumerate` để có thêm chỉ số `i`.

In [4]:
for i, (a, b) in enumerate(zip("bank", "kanb")):
    mark = "KHÁC" if a != b else "giống"
    print(i, a, b, mark)

0 b k KHÁC
1 a a giống
2 n n giống
3 k b KHÁC


### 2.4. `collections.Counter` — đếm số lần xuất hiện

`Counter("bank")` là một dict đặc biệt `{ký tự: số lần}`. Hai chuỗi là **hoán vị** của nhau khi `Counter` của chúng bằng nhau. Một lần swap không thay đổi `Counter`.

In [5]:
from collections import Counter
print(Counter("bank"))
print(Counter("kanb"))
print("bằng nhau?", Counter("bank") == Counter("kanb"))
print("attack vs defend:", Counter("attack") == Counter("defend"))

Counter({'b': 1, 'a': 1, 'n': 1, 'k': 1})
Counter({'k': 1, 'a': 1, 'n': 1, 'b': 1})
bằng nhau? True
attack vs defend: False


## 3. Cách trực giác — Brute force thử mọi swap

Cách ai cũng nghĩ ra: nếu `s1 == s2` → `True`. Nếu không, thử **mọi** cặp `i < j` trong `s2`, swap rồi so sánh với `s1`.

(Swap trên `s1` hay `s2` là như nhau: swap `s2` thành `s1` ⇔ swap `s1` thành `s2` tại cùng hai chỉ số.)

In [6]:
def are_almost_equal_brute(s1: str, s2: str) -> bool:
    if s1 == s2:
        return True
    n = len(s2)
    for i in range(n):
        for j in range(i + 1, n):
            chars = list(s2)
            chars[i], chars[j] = chars[j], chars[i]
            if "".join(chars) == s1:
                return True
    return False

for s1, s2, expected in examples:
    print(s1, s2, "->", are_almost_equal_brute(s1, s2), "| kỳ vọng", expected)

bank kanb -> True | kỳ vọng True
attack defend -> False | kỳ vọng False
kelb kelb -> True | kỳ vọng True


### 3.1. Chi phí của brute force

- Có khoảng `n(n-1)/2` cặp `(i, j)`.
- Mỗi cặp: copy chuỗi `O(n)` và so sánh `O(n)`.
- Tổng: `O(n³)`. Với `n = 100` là ~ 500.000 phép — vẫn qua, nhưng nếu `n = 10^5` thì không thể.

Đếm thử số lần swap phải làm trong trường hợp xấu nhất:

In [7]:
def count_swaps_tried(s1, s2):
    tried = 0
    n = len(s2)
    for i in range(n):
        for j in range(i + 1, n):
            tried += 1
    return tried

for n in [4, 10, 100]:
    print(f"n = {n:3}: số cặp phải thử = {count_swaps_tried('a'*n, 'b'*n)}")

n =   4: số cặp phải thử = 6
n =  10: số cặp phải thử = 45
n = 100: số cặp phải thử = 4950


## 4. Nút thắt và insight

Nút thắt: brute force thử cả những swap **chắc chắn vô ích**. Swap hai vị trí mà `s1` và `s2` đã giống nhau thì tạo ra chỗ khác mới!

Hãy nhìn tập chỉ số `diff = {i : s1[i] != s2[i]}`:

- Một swap chỉ thay đổi **2 vị trí** → sửa được tối đa 2 chỗ khác.
- `len(diff) == 0` → đã bằng → `True`.
- `len(diff) == 1` → một vị trí khác, swap nó với bất kỳ vị trí nào khác sẽ làm hỏng vị trí kia (hoặc vẫn sai) → `False`.
- `len(diff) >= 3` → không thể → `False`.
- `len(diff) == 2`, gọi là `i, j` → swap `i, j` trong `s2` thành công **⇔** `s1[i] == s2[j]` **và** `s1[j] == s2[i]` (hai cặp ký tự "chéo nhau").

In [8]:
def diff_indices(s1, s2):
    return [i for i in range(len(s1)) if s1[i] != s2[i]]

for s1, s2 in [("bank", "kanb"), ("attack", "defend"), ("kelb", "kelb"), ("aa", "ac"), ("ab", "cd")]:
    d = diff_indices(s1, s2)
    print(f"{s1!r:9}{s2!r:9} diff = {d} (số lượng {len(d)})")

'bank'   'kanb'    diff = [0, 3] (số lượng 2)
'attack' 'defend'  diff = [0, 1, 2, 3, 4, 5] (số lượng 6)
'kelb'   'kelb'    diff = [] (số lượng 0)
'aa'     'ac'      diff = [1] (số lượng 1)
'ab'     'cd'      diff = [0, 1] (số lượng 2)


### 4.1. Vì sao cần kiểm tra "chéo", không chỉ đếm 2?

`s1 = "ab"`, `s2 = "cd"`: khác đúng 2 vị trí, nhưng swap `"cd"` → `"dc"` vẫn khác `"ab"`. Vậy **đếm đúng 2 là chưa đủ**, phải kiểm tra `s1[i] == s2[j] and s1[j] == s2[i]`.

In [9]:
s1, s2 = "ab", "cd"
i, j = 0, 1
print("s1[i]==s2[j]?", s1[i] == s2[j], "| s1[j]==s2[i]?", s1[j] == s2[i])
s1, s2 = "bank", "kanb"
i, j = 0, 3
print("bank/kanb chéo:", s1[i] == s2[j], s1[j] == s2[i])

s1[i]==s2[j]? False | s1[j]==s2[i]? False
bank/kanb chéo: True True


## 5. Lời giải chính — thu thập chỉ số khác nhau

**Bước 1.** Duyệt `i` từ `0` tới `n-1`, gặp `s1[i] != s2[i]` thì thêm `i` vào list `diff`.

**Bước 2 (dừng sớm).** Nếu `len(diff) > 2` → trả `False` ngay, không cần duyệt tiếp.

**Bước 3.** Sau vòng lặp:
- `len(diff) == 0` → `True`.
- `len(diff) == 2` và chéo khớp → `True`.
- còn lại (`len(diff) == 1` hoặc 2 nhưng không chéo) → `False`.

In [10]:
# Bước 1 + 2 riêng lẻ, quan sát list diff lớn dần
s1, s2 = "attack", "defend"
diff = []
for i in range(len(s1)):
    if s1[i] != s2[i]:
        diff.append(i)
        print("thêm", i, "-> diff =", diff)
        if len(diff) > 2:
            print("đã > 2 chỗ khác, dừng sớm!")
            break

thêm 0 -> diff = [0]
thêm 1 -> diff = [0, 1]
thêm 2 -> diff = [0, 1, 2]
đã > 2 chỗ khác, dừng sớm!


In [11]:
# Ghép thành hàm hoàn chỉnh
def are_almost_equal(s1: str, s2: str) -> bool:
    diff = []
    for i in range(len(s1)):
        if s1[i] != s2[i]:
            diff.append(i)
            if len(diff) > 2:
                return False
    if len(diff) == 0:
        return True
    if len(diff) != 2:
        return False
    i, j = diff
    return s1[i] == s2[j] and s1[j] == s2[i]

for s1, s2, expected in examples + [("aa", "ac", False), ("ab", "cd", False)]:
    print(f"{s1!r:9}{s2!r:9} -> {are_almost_equal(s1, s2)} | kỳ vọng {expected}")

'bank'   'kanb'    -> True | kỳ vọng True
'attack' 'defend'  -> False | kỳ vọng False
'kelb'   'kelb'    -> True | kỳ vọng True
'aa'     'ac'      -> False | kỳ vọng False
'ab'     'cd'      -> False | kỳ vọng False


`i, j = diff` là **unpacking**: list 2 phần tử được tách ra hai biến. Nếu `diff` không đúng 2 phần tử sẽ lỗi `ValueError` — nên ta phải kiểm tra `len(diff) == 2` trước.

## 6. TRACE từng bước

Trace `s1 = "bank"`, `s2 = "kanb"`:

| i | s1[i] | s2[i] | khác? | diff sau |
|---|---|---|---|---|
| 0 | b | k | có | [0] |
| 1 | a | a | không | [0] |
| 2 | n | n | không | [0] |
| 3 | k | b | có | [0, 3] |

Cuối: `len(diff) == 2`, `i=0, j=3`: `s1[0]='b' == s2[3]='b'` ✓, `s1[3]='k' == s2[0]='k'` ✓ → `True`.

In [12]:
def trace_are_almost_equal(s1: str, s2: str) -> bool:
    print(f"=== s1={s1!r}, s2={s2!r} ===")
    diff = []
    for i in range(len(s1)):
        differ = s1[i] != s2[i]
        if differ:
            diff.append(i)
        print(f"i={i}: s1[i]={s1[i]}, s2[i]={s2[i]}, khác={differ}, diff={diff}")
        if len(diff) > 2:
            print("-> hơn 2 chỗ khác: return False")
            return False
    if len(diff) == 0:
        print("-> không có chỗ khác: return True")
        return True
    if len(diff) != 2:
        print(f"-> có {len(diff)} chỗ khác (không phải 2): return False")
        return False
    i, j = diff
    ok = s1[i] == s2[j] and s1[j] == s2[i]
    print(f"-> kiểm tra chéo: s1[{i}]={s1[i]} vs s2[{j}]={s2[j]}, s1[{j}]={s1[j]} vs s2[{i}]={s2[i]} -> {ok}")
    return ok

trace_are_almost_equal("bank", "kanb")
print()
trace_are_almost_equal("attack", "defend")

=== s1='bank', s2='kanb' ===
i=0: s1[i]=b, s2[i]=k, khác=True, diff=[0]
i=1: s1[i]=a, s2[i]=a, khác=False, diff=[0]
i=2: s1[i]=n, s2[i]=n, khác=False, diff=[0]
i=3: s1[i]=k, s2[i]=b, khác=True, diff=[0, 3]
-> kiểm tra chéo: s1[0]=b vs s2[3]=b, s1[3]=k vs s2[0]=k -> True

=== s1='attack', s2='defend' ===
i=0: s1[i]=a, s2[i]=d, khác=True, diff=[0]
i=1: s1[i]=t, s2[i]=e, khác=True, diff=[0, 1]
i=2: s1[i]=t, s2[i]=f, khác=True, diff=[0, 1, 2]
-> hơn 2 chỗ khác: return False


False

### 6.1. Trace các nhánh đặc biệt: 1 chỗ khác, 2 chỗ khác nhưng không chéo

In [13]:
trace_are_almost_equal("aa", "ac")
print()
trace_are_almost_equal("abcd", "abdc")
print()
trace_are_almost_equal("ab", "cd")

=== s1='aa', s2='ac' ===
i=0: s1[i]=a, s2[i]=a, khác=False, diff=[]
i=1: s1[i]=a, s2[i]=c, khác=True, diff=[1]
-> có 1 chỗ khác (không phải 2): return False

=== s1='abcd', s2='abdc' ===
i=0: s1[i]=a, s2[i]=a, khác=False, diff=[]
i=1: s1[i]=b, s2[i]=b, khác=False, diff=[]
i=2: s1[i]=c, s2[i]=d, khác=True, diff=[2]
i=3: s1[i]=d, s2[i]=c, khác=True, diff=[2, 3]
-> kiểm tra chéo: s1[2]=c vs s2[3]=c, s1[3]=d vs s2[2]=d -> True

=== s1='ab', s2='cd' ===
i=0: s1[i]=a, s2[i]=c, khác=True, diff=[0]
i=1: s1[i]=b, s2[i]=d, khác=True, diff=[0, 1]
-> kiểm tra chéo: s1[0]=a vs s2[1]=d, s1[1]=b vs s2[0]=c -> False


False

## 7. Các lời giải khác

### Cách 2 — `zip` + list comprehension (gọn)

Lấy các **cặp ký tự** khác nhau thay vì chỉ số. Với 2 cặp `(a, b)` và `(c, d)`, chúng chéo nhau khi `(a, b) == (d, c)`, tức cặp thứ hai đảo ngược bằng cặp thứ nhất.

In [14]:
def are_almost_equal_zip(s1: str, s2: str) -> bool:
    pairs = [(a, b) for a, b in zip(s1, s2) if a != b]
    if not pairs:
        return True
    return len(pairs) == 2 and pairs[0] == pairs[1][::-1]

print(are_almost_equal_zip("bank", "kanb"), are_almost_equal_zip("ab", "cd"))
print("đảo tuple:", ("b", "k")[::-1])

True False
đảo tuple: ('k', 'b')


### Cách 3 — `Counter` + đếm số chỗ khác

Điều kiện tương đương: hai chuỗi là hoán vị của nhau (`Counter` bằng nhau) **và** số chỗ khác ∈ {0, 2}.

Vì sao đủ? Nếu khác đúng 2 chỗ `i, j` và multiset ký tự bằng nhau thì sau khi bỏ các vị trí giống nhau, `{s1[i], s1[j]}` = `{s2[i], s2[j]}` dưới dạng multiset, mà `s1[i] != s2[i]` nên buộc `s1[i] == s2[j]`, `s1[j] == s2[i]`.

In [15]:
from collections import Counter

def are_almost_equal_counter(s1: str, s2: str) -> bool:
    diff_count = sum(1 for a, b in zip(s1, s2) if a != b)
    return diff_count in (0, 2) and Counter(s1) == Counter(s2)

for s1, s2 in [("bank", "kanb"), ("ab", "cd"), ("aa", "ac"), ("kelb", "kelb")]:
    print(s1, s2, "->", are_almost_equal_counter(s1, s2))

bank kanb -> True
ab cd -> False
aa ac -> False
kelb kelb -> True


`sum(1 for ... if ...)` là **generator expression**: đếm số phần tử thỏa điều kiện mà không tạo list. `x in (0, 2)` kiểm tra `x` có thuộc tuple `(0, 2)`.

### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử mọi swap | `O(n³)` | `O(n)` | dễ nghĩ, chậm |
| 2. Chỉ số khác (chính) | `diff` có 0 hoặc 2 phần tử chéo | `O(n)` | `O(1)` (tối đa 3 phần tử) | dừng sớm |
| 3. `zip` cặp ký tự | so cặp đảo | `O(n)` | `O(n)` worst | rất gọn |
| 4. `Counter` | hoán vị + đếm 0/2 | `O(n)` | `O(1)` (26 chữ) | cần chứng minh thêm |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def areAlmostEqual(self, s1: str, s2: str) -> bool:
        diff = []
        for i in range(len(s1)):
            if s1[i] != s2[i]:
                diff.append(i)
                if len(diff) > 2:
                    return False
        if not diff:
            return True
        if len(diff) != 2:
            return False
        i, j = diff
        return s1[i] == s2[j] and s1[j] == s2[i]

sol = Solution()
for s1, s2, expected in examples:
    print(sol.areAlmostEqual(s1, s2), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


## 9. Vì sao thuật toán đúng?

Gọi `D` là tập chỉ số mà `s1[i] != s2[i]`.

- Một swap `(p, q)` trên `s2` chỉ đổi giá trị ở **hai** vị trí `p, q`. Mọi vị trí ngoài `{p, q}` giữ nguyên trạng thái khác/giống. Do đó để hết khác, phải có `D ⊆ {p, q}` → `|D| <= 2`.
- `|D| = 0`: swap `(i, i)` (không đổi) → `True`.
- `|D| = 1`, `D = {i}`: phải có `i ∈ {p, q}`. Nếu `p = q = i` thì không đổi gì, vẫn sai. Nếu `q != i` thì vị trí `q` đang đúng (`s2[q] == s1[q]`), sau swap nó nhận `s2[i]`; để đúng cần `s2[i] == s1[q] == s2[q]`, khi đó vị trí `i` nhận `s2[q] == s2[i] != s1[i]` → vẫn sai. → `False`.
- `|D| = 2`, `D = {i, j}`: buộc `{p, q} = {i, j}`; sau swap cần `s2[j] == s1[i]` và `s2[i] == s1[j]` — đúng điều kiện code kiểm tra.

Invariant vòng lặp: sau khi xét `s1[0..k]`, `diff` chứa đúng các chỉ số khác nhau trong đoạn đó (hoặc ta đã dừng vì > 2, khi đó đáp án chắc chắn `False`).

## 10. Độ phức tạp

`n = len(s1)`.

- **Time:** `O(n)` — một lượt duyệt, có thể dừng sớm.
- **Space:** `O(1)` — `diff` chứa tối đa 3 chỉ số.

So với brute force `O(n³)`, đây là bước nhảy lớn nhờ chỉ quan tâm vị trí **khác nhau**.

## 11. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.areAlmostEqual("bank", "kanb"), True)
check("đề 2", sol.areAlmostEqual("attack", "defend"), False)
check("đề 3", sol.areAlmostEqual("kelb", "kelb"), True)
check("1 ký tự giống", sol.areAlmostEqual("a", "a"), True)
check("1 ký tự khác", sol.areAlmostEqual("a", "b"), False)
check("1 chỗ khác", sol.areAlmostEqual("aa", "ac"), False)
check("2 chỗ không chéo", sol.areAlmostEqual("ab", "cd"), False)
check("2 chỗ chéo liền kề", sol.areAlmostEqual("abcd", "abdc"), True)
check("3 chỗ khác (vòng)", sol.areAlmostEqual("abc", "bca"), False)
check("hoán vị nhưng 4 chỗ", sol.areAlmostEqual("abcd", "badc"), False)
check("ký tự lặp", sol.areAlmostEqual("aabb", "abab"), True)

PASS đề 1: True
PASS đề 2: False
PASS đề 3: True
PASS 1 ký tự giống: True
PASS 1 ký tự khác: False
PASS 1 chỗ khác: False
PASS 2 chỗ không chéo: False
PASS 2 chỗ chéo liền kề: True
PASS 3 chỗ khác (vòng): False
PASS hoán vị nhưng 4 chỗ: False
PASS ký tự lặp: True


In [18]:
import random
random.seed(1790)

solvers = [are_almost_equal_brute, are_almost_equal, are_almost_equal_zip,
           are_almost_equal_counter, Solution().areAlmostEqual]

for _ in range(3000):
    n = random.randint(1, 7)
    s1 = "".join(random.choice("abc") for _ in range(n))
    mode = random.random()
    if mode < 0.4:      # tạo s2 bằng một swap ngẫu nhiên -> đáp án phải True
        chars = list(s1)
        i, j = random.randrange(n), random.randrange(n)
        chars[i], chars[j] = chars[j], chars[i]
        s2 = "".join(chars)
    else:
        s2 = "".join(random.choice("abc") for _ in range(n))
    results = [f(s1, s2) for f in solvers]
    assert len(set(results)) == 1, (s1, s2, results)
print("Đối chiếu", len(solvers), "cách trên 3000 cặp ngẫu nhiên (seed 1790): khớp hoàn toàn")

Đối chiếu 5 cách trên 3000 cặp ngẫu nhiên (seed 1790): khớp hoàn toàn


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra hoán vị (`sorted` hoặc `Counter`)

```python
return sorted(s1) == sorted(s2)
```

Sai với `"abc"` / `"bca"`: là hoán vị nhưng cần 2 swap.

In [19]:
def wrong_sorted(s1, s2):
    return sorted(s1) == sorted(s2)

print("abc/bca sai:", wrong_sorted("abc", "bca"), "| đúng:", are_almost_equal("abc", "bca"))

abc/bca sai: True | đúng: False


### Lỗi 2: Chỉ đếm số chỗ khác bằng 2

```python
return diff_count in (0, 2)
```

Sai với `"ab"` / `"cd"`. Phải kiểm tra chéo (hoặc kèm `Counter`).

### Lỗi 3: Quên trường hợp đã bằng nhau

Nếu viết `return len(diff) == 2 and ...` thì `"kelb"/"kelb"` trả `False` sai.

In [20]:
def wrong_count_only(s1, s2):
    return sum(a != b for a, b in zip(s1, s2)) in (0, 2)

def wrong_forget_equal(s1, s2):
    diff = [i for i in range(len(s1)) if s1[i] != s2[i]]
    return len(diff) == 2 and s1[diff[0]] == s2[diff[1]] and s1[diff[1]] == s2[diff[0]]

print("ab/cd  đếm-only:", wrong_count_only("ab", "cd"), "| đúng:", are_almost_equal("ab", "cd"))
print("kelb   quên-bằng:", wrong_forget_equal("kelb", "kelb"), "| đúng:", are_almost_equal("kelb", "kelb"))

ab/cd  đếm-only: True | đúng: False
kelb   quên-bằng: False | đúng: True


### Lỗi 4: Unpack `i, j = diff` khi `diff` có 1 phần tử

`i, j = [3]` → `ValueError: not enough values to unpack`. Luôn kiểm tra `len(diff) == 2` trước.

### Lỗi 5: Cố sửa chuỗi trực tiếp `s2[i], s2[j] = s2[j], s2[i]`

`TypeError: 'str' object does not support item assignment`. Phải đổi sang `list`.

In [21]:
try:
    i, j = [3]
except ValueError as e:
    print("ValueError:", e)

ValueError: not enough values to unpack (expected 2, got 1)


## 13. Mẹo nhận dạng pattern

- "Hai chuỗi/mảng **cùng độ dài**, so sánh theo vị trí" → duyệt song song bằng `zip`, gom **vị trí khác nhau**.
- "**Một** thao tác chỉ ảnh hưởng `k` vị trí" → số chỗ khác phải `<= k` → loại sớm.
- "Swap được thành nhau" → điều kiện cần là cùng multiset ký tự (`Counter`).
- Bài liên quan: 859 Buddy Strings (bắt buộc đúng 1 swap, có xử lý ký tự lặp), 1657 Determine if Two Strings Are Close, 242 Valid Anagram.

## 14. Biến thể

### 14.1. Buddy Strings (LeetCode 859): **bắt buộc** đúng một swap

Khác biệt: khi `s1 == s2`, cần swap hai vị trí **khác nhau** mà chuỗi không đổi → cần có ký tự lặp lại.

In [22]:
def buddy_strings(s: str, goal: str) -> bool:
    if len(s) != len(goal):
        return False
    if s == goal:
        return len(set(s)) < len(s)       # có ký tự lặp thì swap 2 ký tự giống nhau
    diff = [i for i in range(len(s)) if s[i] != goal[i]]
    return len(diff) == 2 and s[diff[0]] == goal[diff[1]] and s[diff[1]] == goal[diff[0]]

print(buddy_strings("ab", "ba"), buddy_strings("ab", "ab"), buddy_strings("aa", "aa"))

True False True


### 14.2. Số swap tối thiểu để biến `s2` thành `s1` (cùng multiset)

Thử tham lam: duyệt từ trái, gặp vị trí sai thì tìm vị trí `j > i` có `s2[j] == s1[i]` (ưu tiên chỗ mà `s2[i] == s1[j]` để sửa 2 chỗ một lúc) và swap. Với bảng chữ nhỏ đây là bài nâng cao (liên quan chu trình hoán vị — LeetCode 854 K-Similar Strings).

In [23]:
def min_swaps_greedy(s1: str, s2: str) -> int:
    t = list(s2)
    swaps = 0
    for i in range(len(t)):
        if t[i] == s1[i]:
            continue
        best = None
        for j in range(i + 1, len(t)):
            if t[j] == s1[i] and t[j] != s1[j]:
                best = j
                if t[i] == s1[j]:      # swap sửa được cả 2 vị trí
                    break
        t[i], t[best] = t[best], t[i]
        swaps += 1
    return swaps

print(min_swaps_greedy("bank", "kanb"), min_swaps_greedy("abc", "bca"))

1 2


(Tham lam này đúng trong các ví dụ trên nhưng **không đảm bảo tối ưu** cho mọi input — bài 854 cần BFS. Đây là ví dụ cho thấy "biến thể nhỏ" có thể khó hơn hẳn.)

## 15. Bài tập luyện tập

**Bài 1 (tính tay):** Với `s1 = "converse"`, `s2 = "conserve"`, liệt kê `diff` và cho biết kết quả.

**Bài 2 (đoán output):** `list(zip("abc", "xbz"))` in ra gì? `[a != b for a, b in zip("abc", "xbz")]` in ra gì?

**Bài 3 (điền TODO):** Hoàn thiện hàm thu thập chỉ số khác nhau có dừng sớm.

In [24]:
def ex3_diff_upto3(s1: str, s2: str) -> list:
    # TODO 1: diff = []
    # TODO 2: duyệt i, nếu s1[i] != s2[i] thì append
    # TODO 3: nếu len(diff) == 3 thì return diff ngay
    pass

# assert ex3_diff_upto3("bank", "kanb") == [0, 3]
# assert ex3_diff_upto3("attack", "defend") == [0, 1, 2]

**Bài 4 (tự viết):** Viết lại `areAlmostEqual` nhưng **không dùng list**: chỉ dùng hai biến `first = -1`, `second = -1` và một biến đếm.

In [25]:
def ex4_almost_equal_no_list(s1: str, s2: str) -> bool:
    # TODO: tự viết
    pass

# assert ex4_almost_equal_no_list("bank", "kanb") == True
# assert ex4_almost_equal_no_list("aa", "ac") == False
# assert ex4_almost_equal_no_list("kelb", "kelb") == True

**Bài 5 (biến thể):** Viết `ex5_one_swap_list(a, b)` cho hai **list số nguyên** cùng độ dài: có thể làm bằng nhau với nhiều nhất 1 swap trên một list không?

In [26]:
def ex5_one_swap_list(a: list, b: list) -> bool:
    # TODO
    pass

# assert ex5_one_swap_list([1, 2, 3], [3, 2, 1]) == True
# assert ex5_one_swap_list([1, 2, 3], [2, 3, 1]) == False

**Bài 6 (mini challenge):** Viết `ex6_at_most_two_swaps(s1, s2)` — có thể làm bằng nhau với **nhiều nhất 2 swap** trên `s2` không? (Gợi ý: số chỗ khác `<= 4`, thử mọi swap đầu tiên trong các vị trí khác rồi gọi lại hàm 1 swap.)

In [27]:
def ex6_at_most_two_swaps(s1: str, s2: str) -> bool:
    # TODO
    pass

# assert ex6_at_most_two_swaps("abc", "bca") == True
# assert ex6_at_most_two_swaps("abcd", "badc") == True
# assert ex6_at_most_two_swaps("abcde", "bcdea") == False

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | so từng vị trí | `v/s` ở chỉ số 3, `s/v` ở chỉ số 5 | chéo khớp → True |
| 2 | `zip` ghép cặp | 3 cặp tuple | `[True, False, True]` |
| 3 | giống lời giải chính | `if len(diff) == 3: return diff` | return diff cuối |
| 4 | lưu 2 chỉ số đầu | `count += 1`; `count > 2` thì False | `first`, `second` gán lần lượt |
| 5 | thuật toán giống hệt chuỗi | `zip(a, b)` | so chéo |
| 6 | mỗi swap sửa tối đa 2 | thử swap `(p, q)` với `p, q ∈ diff` | sau đó gọi `are_almost_equal` |

## 17. Đáp án tham khảo

**Bài 1:** `"converse"` vs `"conserve"`: khác tại chỉ số 3 (`v`/`s`) và 5 (`s`/`v`) → chéo khớp → `True`.

**Bài 2:** `[('a','x'), ('b','b'), ('c','z')]` và `[True, False, True]`.

In [28]:
print(diff_indices("converse", "conserve"), are_almost_equal("converse", "conserve"))
print(list(zip("abc", "xbz")), [a != b for a, b in zip("abc", "xbz")])

def ex3_diff_upto3_ans(s1, s2):
    diff = []
    for i in range(len(s1)):
        if s1[i] != s2[i]:
            diff.append(i)
            if len(diff) == 3:
                return diff
    return diff

def ex4_almost_equal_no_list_ans(s1, s2):
    first = second = -1
    count = 0
    for i in range(len(s1)):
        if s1[i] != s2[i]:
            count += 1
            if count == 1:
                first = i
            elif count == 2:
                second = i
            else:
                return False
    if count == 0:
        return True
    if count == 1:
        return False
    return s1[first] == s2[second] and s1[second] == s2[first]

def ex5_one_swap_list_ans(a, b):
    diff = [i for i, (x, y) in enumerate(zip(a, b)) if x != y]
    if not diff:
        return True
    return len(diff) == 2 and a[diff[0]] == b[diff[1]] and a[diff[1]] == b[diff[0]]

def ex6_at_most_two_swaps_ans(s1, s2):
    if are_almost_equal(s1, s2):
        return True
    diff = [i for i in range(len(s1)) if s1[i] != s2[i]]
    if len(diff) > 4:
        return False
    for x in range(len(diff)):
        for y in range(x + 1, len(diff)):
            t = list(s2)
            p, q = diff[x], diff[y]
            t[p], t[q] = t[q], t[p]
            if are_almost_equal(s1, "".join(t)):
                return True
    return False

assert ex3_diff_upto3_ans("bank", "kanb") == [0, 3]
assert ex3_diff_upto3_ans("attack", "defend") == [0, 1, 2]
assert ex4_almost_equal_no_list_ans("bank", "kanb") == True
assert ex4_almost_equal_no_list_ans("aa", "ac") == False
assert ex4_almost_equal_no_list_ans("kelb", "kelb") == True
assert ex5_one_swap_list_ans([1, 2, 3], [3, 2, 1]) == True
assert ex5_one_swap_list_ans([1, 2, 3], [2, 3, 1]) == False
assert ex6_at_most_two_swaps_ans("abc", "bca") == True
assert ex6_at_most_two_swaps_ans("abcd", "badc") == True
assert ex6_at_most_two_swaps_ans("abcde", "bcdea") == False
print("Đáp án tham khảo PASS toàn bộ test!")

[3, 6] True
[('a', 'x'), ('b', 'b'), ('c', 'z')] [True, False, True]
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def areAlmostEqual(self, s1: str, s2: str) -> bool:
        diff = []
        for i in range(len(s1)):
            if s1[i] != s2[i]:
                diff.append(i)
                if len(diff) > 2:
                    return False
        if not diff:
            return True                       # đã bằng nhau
        if len(diff) != 2:
            return False                      # đúng 1 chỗ khác
        i, j = diff
        return s1[i] == s2[j] and s1[j] == s2[i]   # chéo khớp
```

**Câu thần chú:** *"Một swap sửa tối đa 2 chỗ → đếm chỗ khác: 0 thì ok, 2 thì kiểm chéo, còn lại thua."*

**Checklist:**

- [ ] Trường hợp `s1 == s2` trả `True`?
- [ ] Trường hợp đúng 1 chỗ khác trả `False`?
- [ ] 2 chỗ khác có kiểm tra **chéo**, không chỉ đếm?
- [ ] Không unpack `diff` khi nó không có đúng 2 phần tử?
- [ ] Tên method `areAlmostEqual`, trả `bool`?